# 성능이 0.977로 매우 높게 나오게 된 근거 파악

##### 1. 단일 변수 모델 실험(11개 각각)

In [ ]:
# ============================================================
# LightGBM 조합3 단일 변수 모델 실험
# - 데이터 불러오기부터 결과 저장까지 한 번에 실행
# - 기존 시간순 3-Fold 중 마지막 Fold만 사용
# - 총 11개 모델 학습
# ============================================================

import os
import gc
import time
import warnings

import numpy as np
import pandas as pd

from lightgbm import (
    LGBMClassifier,
    early_stopping,
    log_evaluation
)

from sklearn.metrics import average_precision_score


# 경고 메시지 간소화
warnings.filterwarnings("ignore")


# ============================================================
# 1. 기본 설정
# ============================================================

TARGET = "is_fraud"
TIME_COL = "trans_date_trans_time"
RANDOM_STATE = 42
EARLY_STOPPING_ROUNDS = 100

# 기존 조합3 전체 모델의 Fold 3 PR-AUC
FULL_MODEL_FOLD3_PR_AUC = 0.977638


# ============================================================
# 2. 조합3 최종 변수 11개
# ============================================================

SINGLE_FEATURES = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",
    "high_speed"
]


# ============================================================
# 3. 데이터 경로
# ============================================================

DATA_PATH = (
    r"C:\Users\splen\OneDrive\Desktop\BDAI_"
    r"\BOOSTMAP\Fraud-FDS-Project"
    r"\data\fraud_full_features.csv"
)

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"파일을 찾을 수 없습니다:\n{DATA_PATH}"
    )

print("데이터 경로:", DATA_PATH)


# ============================================================
# 4. 필요한 컬럼만 불러오기
# ============================================================

required_columns = (
    [TIME_COL, TARGET]
    + SINGLE_FEATURES
)

df = pd.read_csv(
    DATA_PATH,
    usecols=required_columns,
    parse_dates=[TIME_COL]
)

print("데이터 불러오기 완료:", df.shape)


# ============================================================
# 5. 시간순 정렬
# ============================================================

df = (
    df
    .sort_values(
        TIME_COL,
        kind="mergesort"
    )
    .reset_index(drop=True)
)

assert df[TIME_COL].is_monotonic_increasing

print(
    "전체 기간:",
    df[TIME_COL].min(),
    "~",
    df[TIME_COL].max()
)

print(
    "전체 이상거래:",
    f"{int(df[TARGET].sum()):,}"
)

print(
    "전체 이상거래율:",
    f"{df[TARGET].mean():.4%}"
)


# ============================================================
# 6. 기존 시간순 3-Fold 중 Fold 3 복원
#
# 전체 데이터를 6개 구간으로 분할
# 구간 1~5: Train
# 구간 6: Validation
# ============================================================

boundaries = np.linspace(
    0,
    len(df),
    7,
    dtype=int
)

train_end = boundaries[5]
validation_start = boundaries[5]
validation_end = boundaries[6]

train_df = (
    df
    .iloc[:train_end]
    .copy()
    .reset_index(drop=True)
)

validation_df = (
    df
    .iloc[validation_start:validation_end]
    .copy()
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("시간순 Fold 3 데이터")
print("=" * 70)

print(
    "Train 기간:",
    train_df[TIME_COL].min(),
    "~",
    train_df[TIME_COL].max()
)

print(
    "Validation 기간:",
    validation_df[TIME_COL].min(),
    "~",
    validation_df[TIME_COL].max()
)

print("Train 거래 수:", f"{len(train_df):,}")
print(
    "Validation 거래 수:",
    f"{len(validation_df):,}"
)

print(
    "Validation 이상거래:",
    f"{int(validation_df[TARGET].sum()):,}"
)

print(
    "Validation 이상거래율:",
    f"{validation_df[TARGET].mean():.4%}"
)


# ============================================================
# 7. Train 기준 scale_pos_weight 계산
# ============================================================

negative_count = int(
    (train_df[TARGET] == 0).sum()
)

positive_count = int(
    (train_df[TARGET] == 1).sum()
)

if positive_count == 0:
    raise ValueError(
        "Train에 이상거래가 없어 가중치를 계산할 수 없습니다."
    )

scale_pos_weight = (
    negative_count / positive_count
)

print(
    "scale_pos_weight:",
    f"{scale_pos_weight:.6f}"
)


# ============================================================
# 8. 정답 데이터 준비
# ============================================================

y_train = (
    train_df[TARGET]
    .astype(np.int8)
    .reset_index(drop=True)
)

y_validation = (
    validation_df[TARGET]
    .astype(np.int8)
    .reset_index(drop=True)
)


# ============================================================
# 9. 11개 단일 변수 모델 학습
# ============================================================

single_feature_results = []

experiment_start_time = time.perf_counter()

for feature_number, feature in enumerate(
    SINGLE_FEATURES,
    start=1
):

    feature_start_time = time.perf_counter()

    print("\n" + "=" * 70)
    print(
        f"[{feature_number}/{len(SINGLE_FEATURES)}] "
        f"단일 변수: {feature}"
    )
    print("=" * 70)

    # 해당 변수 하나만 사용
    X_train = (
        train_df[[feature]]
        .copy()
        .reset_index(drop=True)
    )

    X_validation = (
        validation_df[[feature]]
        .copy()
        .reset_index(drop=True)
    )

    categorical_features = []

    # category만 LightGBM 범주형으로 처리
    if feature == "category":

        train_category_levels = (
            X_train["category"]
            .astype("string")
            .fillna("missing")
            .unique()
            .tolist()
        )

        X_train["category"] = pd.Categorical(
            X_train["category"]
            .astype("string")
            .fillna("missing"),
            categories=train_category_levels
        )

        X_validation["category"] = pd.Categorical(
            X_validation["category"]
            .astype("string")
            .fillna("missing"),
            categories=train_category_levels
        )

        categorical_features = [
            "category"
        ]

    # 최종 조합3과 동일한 하이퍼파라미터
    model = LGBMClassifier(
        objective="binary",

        n_estimators=2500,
        learning_rate=0.03,

        num_leaves=23,
        max_depth=5,
        min_child_samples=150,

        subsample=0.9,
        subsample_freq=1,
        colsample_bytree=0.9,

        reg_alpha=0.1,
        reg_lambda=1.0,

        min_split_gain=0.0,
        max_bin=255,

        scale_pos_weight=scale_pos_weight,

        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(
        X_train,
        y_train,

        eval_set=[
            (
                X_validation,
                y_validation
            )
        ],

        eval_metric="average_precision",

        categorical_feature=categorical_features,

        callbacks=[
            early_stopping(
                stopping_rounds=EARLY_STOPPING_ROUNDS,
                first_metric_only=True,
                verbose=False
            ),

            log_evaluation(period=0)
        ]
    )

    validation_probability = (
        model.predict_proba(
            X_validation,
            num_iteration=model.best_iteration_
        )[:, 1]
    )

    single_pr_auc = average_precision_score(
        y_validation,
        validation_probability
    )

    elapsed_seconds = (
        time.perf_counter()
        - feature_start_time
    )

    single_feature_results.append({
        "feature": feature,
        "single_feature_pr_auc":
            single_pr_auc,
        "full_model_pr_auc":
            FULL_MODEL_FOLD3_PR_AUC,
        "difference_from_full":
            (
                single_pr_auc
                - FULL_MODEL_FOLD3_PR_AUC
            ),
        "best_iteration":
            int(model.best_iteration_),
        "elapsed_seconds":
            elapsed_seconds
    })

    print(
        "단일 변수 PR-AUC:",
        f"{single_pr_auc:.6f}"
    )

    print(
        "전체 모델과 차이:",
        f"{single_pr_auc - FULL_MODEL_FOLD3_PR_AUC:+.6f}"
    )

    print(
        "Best iteration:",
        model.best_iteration_
    )

    print(
        "실행시간:",
        f"{elapsed_seconds:.1f}초"
    )

    del (
        X_train,
        X_validation,
        validation_probability,
        model
    )

    gc.collect()


# ============================================================
# 10. 결과표 생성
# ============================================================

single_feature_result_df = (
    pd.DataFrame(
        single_feature_results
    )
    .sort_values(
        "single_feature_pr_auc",
        ascending=False
    )
    .reset_index(drop=True)
)

single_feature_result_df.insert(
    0,
    "rank",
    np.arange(
        1,
        len(single_feature_result_df) + 1
    )
)

print("\n")
print("=" * 95)
print("조합3 단일 변수 모델 실험 결과")
print("=" * 95)

display(
    single_feature_result_df.style.format({
        "single_feature_pr_auc": "{:.6f}",
        "full_model_pr_auc": "{:.6f}",
        "difference_from_full": "{:+.6f}",
        "elapsed_seconds": "{:.1f}"
    })
)


# ============================================================
# 11. 결과 CSV 저장
# ============================================================

RESULT_PATH = (
    "lightGBM_combination3_single_feature_results.csv"
)

single_feature_result_df.to_csv(
    RESULT_PATH,
    index=False,
    encoding="utf-8-sig"
)

total_minutes = (
    time.perf_counter()
    - experiment_start_time
) / 60

print(
    "\n조합3 전체 모델 Fold 3 PR-AUC:",
    f"{FULL_MODEL_FOLD3_PR_AUC:.6f}"
)

print(
    "전체 실행시간:",
    f"{total_minutes:.2f}분"
)

print(
    "결과 저장 파일:",
    RESULT_PATH
)

print("\n✅ 11개 단일 변수 실험 완료")
print("✅ Test 데이터는 사용하지 않았습니다.")

In [2]:
display(single_feature_result_df)

,rank,feature,single_feature_pr_auc,full_model_pr_auc,difference_from_full,best_iteration,elapsed_seconds
0,1,recent_24h_high_amt_count,0.361495,0.977638,-0.616143,2,3.010326
1,2,amt_to_prior_median_ratio,0.316886,0.977638,-0.660752,86,11.897228
2,3,rolling_sum_amt_1h,0.274854,0.977638,-0.702784,241,20.835846
3,4,amt,0.263503,0.977638,-0.714135,197,17.661462
4,5,amt_zscore_card,0.169351,0.977638,-0.808287,132,14.974703
5,6,prior_normal_median_amt,0.125925,0.977638,-0.851713,14,8.881612
6,7,trans_hour,0.024002,0.977638,-0.953636,86,8.817646
7,8,category,0.013213,0.977638,-0.964425,17,8.843875
8,9,count_30min,0.010917,0.977638,-0.966721,12,6.528396
9,10,age,0.006900,0.977638,-0.970738,169,15.872235


In [3]:
import os
import pandas as pd

print("현재 작업 폴더:", os.getcwd())

result_df = pd.read_csv(
    "lightGBM_combination3_single_feature_results.csv"
)

display(result_df)

현재 작업 폴더: c:\Users\splen\OneDrive\Desktop\BDAI_\BDAI\Modeling


,rank,feature,single_feature_pr_auc,full_model_pr_auc,difference_from_full,best_iteration,elapsed_seconds
0,1,recent_24h_high_amt_count,0.361495,0.977638,-0.616143,2,3.010326
1,2,amt_to_prior_median_ratio,0.316886,0.977638,-0.660752,86,11.897228
2,3,rolling_sum_amt_1h,0.274854,0.977638,-0.702784,241,20.835846
3,4,amt,0.263503,0.977638,-0.714135,197,17.661462
4,5,amt_zscore_card,0.169351,0.977638,-0.808287,132,14.974703
5,6,prior_normal_median_amt,0.125925,0.977638,-0.851713,14,8.881612
6,7,trans_hour,0.024002,0.977638,-0.953636,86,8.817646
7,8,category,0.013213,0.977638,-0.964425,17,8.843875
8,9,count_30min,0.010917,0.977638,-0.966721,12,6.528396
9,10,age,0.006900,0.977638,-0.970738,169,15.872235


##### 2. 변수 제거 실험(Ablation) <- 세진님이 하심

1번 실험과 함께 자세한 분석 내용은 ppt!

##### 3. 원자료에서 변수별 사기율 확인

In [4]:
# ============================================================
# 조합3 원자료 변수별 사기율 분석
# - 원본 Train 데이터 사용
# - $800 Rule 적용하지 않음
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. 기본 정보
# ------------------------------------------------------------

TARGET = "is_fraud"

overall_fraud_rate = df[TARGET].mean()
total_fraud_count = int(df[TARGET].sum())

print("전체 거래:", f"{len(df):,}")
print("전체 사기:", f"{total_fraud_count:,}")
print("전체 사기율:", f"{overall_fraud_rate:.4%}")


# ------------------------------------------------------------
# 2. 공통 요약 함수
# ------------------------------------------------------------

def make_fraud_summary(
    source_df,
    group_values,
    variable_name
):

    analysis_df = pd.DataFrame({
        "group": group_values,
        TARGET: source_df[TARGET].to_numpy()
    })

    # 결측치는 별도 구간으로 표시
    analysis_df["group"] = (
        analysis_df["group"]
        .astype("string")
        .fillna("Missing")
    )

    summary = (
        analysis_df
        .groupby(
            "group",
            observed=True,
            sort=False
        )
        .agg(
            transactions=(TARGET, "size"),
            fraud_count=(TARGET, "sum"),
            fraud_rate=(TARGET, "mean")
        )
        .reset_index()
    )

    summary.insert(
        0,
        "variable",
        variable_name
    )

    summary["normal_count"] = (
        summary["transactions"]
        - summary["fraud_count"]
    )

    summary["transaction_share"] = (
        summary["transactions"]
        / len(source_df)
    )

    summary["fraud_coverage"] = (
        summary["fraud_count"]
        / total_fraud_count
    )

    summary["fraud_rate_lift"] = (
        summary["fraud_rate"]
        / overall_fraud_rate
    )

    return summary[
        [
            "variable",
            "group",
            "transactions",
            "fraud_count",
            "normal_count",
            "transaction_share",
            "fraud_rate",
            "fraud_coverage",
            "fraud_rate_lift"
        ]
    ]


# ------------------------------------------------------------
# 3. 구간 설정
# ------------------------------------------------------------

analysis_groups = {}


# ① category
analysis_groups["category"] = (
    df["category"]
)


# ② 거래금액 amt
analysis_groups["amt"] = pd.cut(
    df["amt"],
    bins=[
        0, 50, 100, 200, 300, 400,
        500, 600, 700, 800, 900,
        1000, 1500, 2000, np.inf
    ],
    labels=[
        "0~50",
        "50~100",
        "100~200",
        "200~300",
        "300~400",
        "400~500",
        "500~600",
        "600~700",
        "700~800",
        "800~900",
        "900~1000",
        "1000~1500",
        "1500~2000",
        "2000+"
    ],
    right=False
)


# ③ 거래 시간
analysis_groups["trans_hour"] = (
    df["trans_hour"]
)


# ④ 연령대
analysis_groups["age"] = pd.cut(
    df["age"],
    bins=[
        0, 20, 30, 40, 50,
        60, 70, 80, 90, 100,
        np.inf
    ],
    labels=[
        "0~19",
        "20~29",
        "30~39",
        "40~49",
        "50~59",
        "60~69",
        "70~79",
        "80~89",
        "90~99",
        "100+"
    ],
    right=False
)


# ⑤ 최근 24시간 $500 이상 거래 횟수
analysis_groups[
    "recent_24h_high_amt_count"
] = pd.cut(
    df["recent_24h_high_amt_count"],
    bins=[
        -0.5, 0.5, 1.5, 2.5,
        3.5, 5.5, 10.5, np.inf
    ],
    labels=[
        "0",
        "1",
        "2",
        "3",
        "4~5",
        "6~10",
        "11+"
    ]
)


# ⑥ 이전 정상거래 중앙값 대비 현재 금액
analysis_groups[
    "amt_to_prior_median_ratio"
] = pd.cut(
    df["amt_to_prior_median_ratio"],
    bins=[
        -np.inf,
        0.5,
        1,
        2,
        5,
        10,
        np.inf
    ],
    labels=[
        "0.5배 미만",
        "0.5~1배",
        "1~2배",
        "2~5배",
        "5~10배",
        "10배 이상"
    ],
    right=False
)


# ⑦ 최근 1시간 누적 거래금액: 동일 건수 10구간
analysis_groups[
    "rolling_sum_amt_1h"
] = pd.qcut(
    df["rolling_sum_amt_1h"],
    q=10,
    duplicates="drop"
)


# ⑧ 고객별 거래금액 Z-score
analysis_groups[
    "amt_zscore_card"
] = pd.cut(
    df["amt_zscore_card"],
    bins=[
        -np.inf,
        -1,
        0,
        1,
        2,
        3,
        5,
        np.inf
    ],
    labels=[
        "-1 미만",
        "-1~0",
        "0~1",
        "1~2",
        "2~3",
        "3~5",
        "5 이상"
    ],
    right=False
)


# ⑨ 이전 정상거래 중앙값: 동일 건수 10구간
analysis_groups[
    "prior_normal_median_amt"
] = pd.qcut(
    df["prior_normal_median_amt"],
    q=10,
    duplicates="drop"
)


# ⑩ 최근 30분 거래 횟수
analysis_groups[
    "count_30min"
] = pd.cut(
    df["count_30min"],
    bins=[
        -0.5,
        0.5,
        1.5,
        2.5,
        3.5,
        5.5,
        10.5,
        np.inf
    ],
    labels=[
        "0",
        "1",
        "2",
        "3",
        "4~5",
        "6~10",
        "11+"
    ]
)


# ⑪ 고속 이동 여부
analysis_groups["high_speed"] = (
    df["high_speed"]
)


# ------------------------------------------------------------
# 4. 변수별 분석 실행
# ------------------------------------------------------------

fraud_summary_by_variable = {}

all_fraud_summaries = []

for variable_name, group_values in analysis_groups.items():

    variable_summary = make_fraud_summary(
        source_df=df,
        group_values=group_values,
        variable_name=variable_name
    )

    fraud_summary_by_variable[
        variable_name
    ] = variable_summary

    all_fraud_summaries.append(
        variable_summary
    )


# ------------------------------------------------------------
# 5. 전체 결과 합치기
# ------------------------------------------------------------

all_variable_fraud_summary = pd.concat(
    all_fraud_summaries,
    ignore_index=True
)


# ------------------------------------------------------------
# 6. 변수별 결과 출력
# ------------------------------------------------------------

for variable_name in analysis_groups.keys():

    print("\n")
    print("=" * 90)
    print(f"{variable_name} 구간별 사기율")
    print("=" * 90)

    display(
        fraud_summary_by_variable[
            variable_name
        ].style.format({
            "transactions": "{:,}",
            "fraud_count": "{:,}",
            "normal_count": "{:,}",
            "transaction_share": "{:.2%}",
            "fraud_rate": "{:.4%}",
            "fraud_coverage": "{:.2%}",
            "fraud_rate_lift": "{:.2f}배"
        })
    )


# ------------------------------------------------------------
# 7. 구간별 위험배수 상위 20개
# ------------------------------------------------------------

high_risk_groups = (
    all_variable_fraud_summary[
        all_variable_fraud_summary[
            "transactions"
        ] >= 100
    ]
    .sort_values(
        "fraud_rate_lift",
        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)

print("\n")
print("=" * 90)
print("전체 변수 중 사기 위험배수 상위 20개 구간")
print("=" * 90)

display(
    high_risk_groups.style.format({
        "transactions": "{:,}",
        "fraud_count": "{:,}",
        "normal_count": "{:,}",
        "transaction_share": "{:.2%}",
        "fraud_rate": "{:.4%}",
        "fraud_coverage": "{:.2%}",
        "fraud_rate_lift": "{:.2f}배"
    })
)


# ------------------------------------------------------------
# 8. CSV 저장
# ------------------------------------------------------------

all_variable_fraud_summary.to_csv(
    "combination3_variable_fraud_rate_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

high_risk_groups.to_csv(
    "combination3_high_risk_groups_top20.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n✅ 조합3 변수별 사기율 분석 완료")
print(
    "전체 결과:",
    "combination3_variable_fraud_rate_summary.csv"
)
print(
    "위험구간 상위 결과:",
    "combination3_high_risk_groups_top20.csv"
)

전체 거래: 1,296,675
전체 사기: 7,506
전체 사기율: 0.5789%


category 구간별 사기율


,variable,group,transactions,fraud_count,normal_count,transaction_share,fraud_rate,fraud_coverage,fraud_rate_lift
0,category,misc_net,"63,287",915,"62,372",4.88%,1.4458%,12.19%,2.50배
1,category,grocery_pos,"123,638","1,743","121,895",9.54%,1.4098%,23.22%,2.44배
2,category,entertainment,"94,014",233,"93,781",7.25%,0.2478%,3.10%,0.43배
3,category,gas_transport,"131,659",618,"131,041",10.15%,0.4694%,8.23%,0.81배
4,category,misc_pos,"79,655",250,"79,405",6.14%,0.3139%,3.33%,0.54배
5,category,grocery_net,"45,452",134,"45,318",3.51%,0.2948%,1.79%,0.51배
6,category,shopping_net,"97,543","1,713","95,830",7.52%,1.7561%,22.82%,3.03배
7,category,shopping_pos,"116,672",843,"115,829",9.00%,0.7225%,11.23%,1.25배
8,category,food_dining,"91,461",151,"91,310",7.05%,0.1651%,2.01%,0.29배
9,category,personal_care,"90,758",220,"90,538",7.00%,0.2424%,2.93%,0.42배




amt 구간별 사기율


,variable,group,transactions,fraud_count,normal_count,transaction_share,fraud_rate,fraud_coverage,fraud_rate_lift
0,amt,0~50,"672,214","1,607","670,607",51.84%,0.2391%,21.41%,0.41배
1,amt,100~200,"173,017",150,"172,867",13.34%,0.0867%,2.00%,0.15배
2,amt,200~300,"31,631",795,"30,836",2.44%,2.5134%,10.59%,4.34배
3,amt,50~100,"389,514",45,"389,469",30.04%,0.0116%,0.60%,0.02배
4,amt,300~400,"8,600","1,160","7,440",0.66%,13.4884%,15.45%,23.30배
5,amt,600~700,"1,945",167,"1,778",0.15%,8.5861%,2.22%,14.83배
6,amt,500~600,"4,558",94,"4,464",0.35%,2.0623%,1.25%,3.56배
7,amt,1000~1500,"2,641",950,"1,691",0.20%,35.9712%,12.66%,62.14배
8,amt,400~500,"6,068",101,"5,967",0.47%,1.6645%,1.35%,2.88배
9,amt,900~1000,"1,602",943,659,0.12%,58.8639%,12.56%,101.69배




trans_hour 구간별 사기율


...



high_speed 구간별 사기율


,variable,group,transactions,fraud_count,normal_count,transaction_share,fraud_rate,fraud_coverage,fraud_rate_lift
0,high_speed,0,"1,139,777","6,142","1,133,635",87.90%,0.5389%,81.83%,0.93배
1,high_speed,1,"156,898","1,364","155,534",12.10%,0.8694%,18.17%,1.50배




전체 변수 중 사기 위험배수 상위 20개 구간


,variable,group,transactions,fraud_count,normal_count,transaction_share,fraud_rate,fraud_coverage,fraud_rate_lift
0,recent_24h_high_amt_count,4~5,"1,140",899,241,0.09%,78.8596%,11.98%,136.23배
1,recent_24h_high_amt_count,3,"1,342","1,047",295,0.10%,78.0179%,13.95%,134.78배
2,recent_24h_high_amt_count,6~10,214,153,61,0.02%,71.4953%,2.04%,123.51배
3,amt,900~1000,"1,602",943,659,0.12%,58.8639%,12.56%,101.69배
4,amt,800~900,"1,679",831,848,0.13%,49.4937%,11.07%,85.50배
5,recent_24h_high_amt_count,2,"3,132","1,520","1,612",0.24%,48.5313%,20.25%,83.84배
6,amt_to_prior_median_ratio,Missing,"1,649",741,908,0.13%,44.9363%,9.87%,77.63배
7,prior_normal_median_amt,Missing,"1,649",741,908,0.13%,44.9363%,9.87%,77.63배
8,amt,1000~1500,"2,641",950,"1,691",0.20%,35.9712%,12.66%,62.14배
9,amt,700~800,"1,910",663,"1,247",0.15%,34.7120%,8.83%,59.97배



✅ 조합3 변수별 사기율 분석 완료
전체 결과: combination3_variable_fraud_rate_summary.csv
위험구간 상위 결과: combination3_high_risk_groups_top20.csv


# Q. 생긴 궁금증 해결하기

조합 3 전체 / age 제외 / prior_normal_median_amt 제외 / 둘 다 제외

총 4개를 마지막 시간 Fold에서 동일 조건으로 비교.

In [5]:
# ============================================================
# 1단계: 라이브러리 및 Train 데이터 불러오기
# ============================================================

import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 데이터 경로
# ------------------------------------------------------------
TRAIN_DATA_PATH = Path(
    r"C:\Users\splen\OneDrive\Desktop\BDAI_\BOOSTMAP"
    r"\Fraud-FDS-Project\data\fraud_full_features.csv"
)

if not TRAIN_DATA_PATH.exists():
    raise FileNotFoundError(f"Train 파일을 찾을 수 없습니다:\n{TRAIN_DATA_PATH}")

# 필요한 컬럼만 읽기
FINAL_FEATURES = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",
    "high_speed"
]

TARGET = "is_fraud"
TIME_COLUMN = "trans_date_trans_time"

use_columns = FINAL_FEATURES + [TARGET, TIME_COLUMN]

df = pd.read_csv(
    TRAIN_DATA_PATH,
    usecols=use_columns,
    parse_dates=[TIME_COLUMN]
)

# 시간순 정렬
df = (
    df.sort_values(TIME_COLUMN)
      .reset_index(drop=True)
)

# LightGBM 범주형 변수 지정
df["category"] = df["category"].astype("category")

print("Train 파일 경로:", TRAIN_DATA_PATH)
print("전체 데이터 크기:", df.shape)
print("전체 사기 건수:", int(df[TARGET].sum()))
print("전체 사기율:", f"{df[TARGET].mean():.4%}")
print("시작 시각:", df[TIME_COLUMN].min())
print("종료 시각:", df[TIME_COLUMN].max())
print("\n✅ 1단계 완료")

Train 파일 경로: C:\Users\splen\OneDrive\Desktop\BDAI_\BOOSTMAP\Fraud-FDS-Project\data\fraud_full_features.csv
전체 데이터 크기: (1296675, 13)
전체 사기 건수: 7506
전체 사기율: 0.5789%
시작 시각: 2019-01-01 00:00:18
종료 시각: 2020-06-21 12:13:37

✅ 1단계 완료


In [8]:
# ============================================================
# 수정 실험: 트리 1,756개로 고정한 Ablation
# ============================================================

feature_sets = {
    "Baseline_11개": FINAL_FEATURES,

    "age_제외_10개": [
        feature for feature in FINAL_FEATURES
        if feature != "age"
    ],

    "prior_median_제외_10개": [
        feature for feature in FINAL_FEATURES
        if feature != "prior_normal_median_amt"
    ],

    "age_prior_median_제외_9개": [
        feature for feature in FINAL_FEATURES
        if feature not in ["age", "prior_normal_median_amt"]
    ]
}

# 마지막 시간 Fold
X_all = df[FINAL_FEATURES].copy()
y_all = df[TARGET].astype(int).copy()

time_split = TimeSeriesSplit(n_splits=3)
train_idx, valid_idx = list(time_split.split(X_all))[-1]

X_train_base = X_all.iloc[train_idx].copy()
X_valid_base = X_all.iloc[valid_idx].copy()

y_train = y_all.iloc[train_idx].copy()
y_valid = y_all.iloc[valid_idx].copy()

negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
scale_pos_weight = negative_count / positive_count

model_params = {
    "objective": "binary",
    "n_estimators": 1756,       # 최종 모델과 동일한 트리 수
    "learning_rate": 0.03,
    "num_leaves": 23,
    "max_depth": 5,
    "min_child_samples": 150,
    "subsample": 0.9,
    "subsample_freq": 1,
    "colsample_bytree": 0.9,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "min_split_gain": 0.0,
    "max_bin": 255,
    "scale_pos_weight": scale_pos_weight,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1
}

results = []
trained_models = {}

for experiment_name, features in feature_sets.items():

    print("\n" + "=" * 70)
    print("실험:", experiment_name)
    print("변수 수:", len(features))
    print(
        "제외 변수:",
        sorted(set(FINAL_FEATURES) - set(features)) or "없음"
    )

    X_train = X_train_base[features].copy()
    X_valid = X_valid_base[features].copy()

    categorical_features = (
        ["category"] if "category" in features else []
    )

    start_time = time.time()

    model = lgb.LGBMClassifier(**model_params)

    # 조기 종료 없이 1,756개 트리 전체 학습
    model.fit(
        X_train,
        y_train,
        categorical_feature=categorical_features
    )

    valid_probability = model.predict_proba(X_valid)[:, 1]

    pr_auc = average_precision_score(
        y_valid,
        valid_probability
    )

    # 모델별 F1 최대 임계값
    precision_values, recall_values, thresholds = (
        precision_recall_curve(y_valid, valid_probability)
    )

    f1_values = (
        2 * precision_values[:-1] * recall_values[:-1]
        / (
            precision_values[:-1]
            + recall_values[:-1]
            + 1e-12
        )
    )

    best_index = int(np.nanargmax(f1_values))
    best_threshold = float(thresholds[best_index])

    valid_prediction = (
        valid_probability >= best_threshold
    ).astype(int)

    precision = precision_score(
        y_valid, valid_prediction, zero_division=0
    )
    recall = recall_score(
        y_valid, valid_prediction, zero_division=0
    )
    f1 = f1_score(
        y_valid, valid_prediction, zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_valid,
        valid_prediction
    ).ravel()

    elapsed_seconds = time.time() - start_time

    results.append({
        "experiment": experiment_name,
        "feature_count": len(features),
        "removed_features": ", ".join(
            sorted(set(FINAL_FEATURES) - set(features))
        ) or "없음",
        "pr_auc": pr_auc,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "tn": int(tn),
        "best_threshold": best_threshold,
        "tree_count": 1756,
        "elapsed_seconds": elapsed_seconds
    })

    trained_models[experiment_name] = model

    print(f"PR-AUC: {pr_auc:.6f}")
    print(f"Precision: {precision:.6f}")
    print(f"Recall: {recall:.6f}")
    print(f"F1-score: {f1:.6f}")
    print(f"FP: {fp:,} / FN: {fn:,}")
    print(f"최적 임계값: {best_threshold:.6f}")
    print(f"소요 시간: {elapsed_seconds:.1f}초")

print("\n✅ 수정 Ablation 학습 완료")


실험: Baseline_11개
변수 수: 11
제외 변수: 없음
PR-AUC: 0.978672
Precision: 0.966022
Recall: 0.931921
F1-score: 0.948665
FP: 65 / FN: 135
최적 임계값: 0.953773
소요 시간: 122.0초

실험: age_제외_10개
변수 수: 10
제외 변수: ['age']
PR-AUC: 0.967799
Precision: 0.935316
Recall: 0.904186
F1-score: 0.919487
FP: 124 / FN: 190
최적 임계값: 0.964670
소요 시간: 93.7초

실험: prior_median_제외_10개
변수 수: 10
제외 변수: ['prior_normal_median_amt']
PR-AUC: 0.977424
Precision: 0.962225
Recall: 0.924861
F1-score: 0.943173
FP: 72 / FN: 149
최적 임계값: 0.974076
소요 시간: 89.8초

실험: age_prior_median_제외_9개
변수 수: 9
제외 변수: ['age', 'prior_normal_median_amt']
PR-AUC: 0.961840
Precision: 0.924934
Recall: 0.888553
F1-score: 0.906379
FP: 143 / FN: 221
최적 임계값: 0.983010
소요 시간: 85.8초

✅ 수정 Ablation 학습 완료


In [9]:
# ============================================================
# 수정 결과표
# ============================================================

result_df = pd.DataFrame(results)

baseline = result_df.loc[
    result_df["experiment"] == "Baseline_11개"
].iloc[0]

result_df["pr_auc_difference"] = (
    result_df["pr_auc"] - baseline["pr_auc"]
)

result_df["f1_difference"] = (
    result_df["f1_score"] - baseline["f1_score"]
)

result_df["fp_difference"] = (
    result_df["fp"] - baseline["fp"]
)

result_df["fn_difference"] = (
    result_df["fn"] - baseline["fn"]
)

display_columns = [
    "experiment",
    "feature_count",
    "removed_features",
    "pr_auc",
    "pr_auc_difference",
    "precision",
    "recall",
    "f1_score",
    "f1_difference",
    "fp",
    "fp_difference",
    "fn",
    "fn_difference",
    "best_threshold",
    "tree_count",
    "elapsed_seconds"
]

result_df = (
    result_df[display_columns]
    .sort_values("pr_auc", ascending=False)
    .reset_index(drop=True)
)

display(result_df.round(6))

result_df.to_csv(
    "combination3_age_prior_median_ablation_fixed.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✅ 수정 결과 저장 완료")

,experiment,feature_count,removed_features,pr_auc,pr_auc_difference,precision,recall,f1_score,f1_difference,fp,fp_difference,fn,fn_difference,best_threshold,tree_count,elapsed_seconds
0,Baseline_11개,11,없음,0.978672,0.000000,0.966022,0.931921,0.948665,0.000000,65,0,135,0,0.953773,1756,121.961001
1,prior_median_제외_10개,10,prior_normal_median_amt,0.977424,-0.001248,0.962225,0.924861,0.943173,-0.005492,72,7,149,14,0.974076,1756,89.822699
2,age_제외_10개,10,age,0.967799,-0.010873,0.935316,0.904186,0.919487,-0.029178,124,59,190,55,0.964670,1756,93.661162
3,age_prior_median_제외_9개,9,"age, prior_normal_median_amt",0.961840,-0.016832,0.924934,0.888553,0.906379,-0.042287,143,78,221,86,0.983010,1756,85.796686


✅ 수정 결과 저장 완료
